In [15]:
import numpy as np
from openfermion import get_fermion_operator

from ofex.clifford import str_tableau, tableau_to_pauli
from ofex.linalg.sparse_tools import diagonalization
from ofex.state.chem_ref_state import hf_ground, cisd_ground
from ofex.state.state_tools import state_allclose
from ofex.transforms import fermion_to_qubit_state, fermion_to_qubit_operator, find_pauli_symmetry, \
    qubit_reduction_operator, qubit_reduction_state
from ofex.utils.chem import molecule_example, run_driver


## 1. Prepare Hamiltonian

For the details of this tutorial, refer to [arXiv:1701.08213](https://arxiv.org/abs/1701.08213).

In [16]:
# Obtain the Fermionic Hamiltonian

mol_name = "H4"

mol = molecule_example(mol_name)
mol = run_driver(mol, run_cisd=True)
fham = mol.get_molecular_hamiltonian()
fham = get_fermion_operator(fham)
f_const = fham.constant
fham = fham - f_const

In [17]:
# Fermion-to-Qubit mapping

transform = "bravyi_kitaev"
f2q_kwargs = {"n_qubits": mol.n_qubits}

n_qubits = mol.n_qubits

hf_state_fermion = hf_ground(mol)
hf_state =fermion_to_qubit_state(hf_state_fermion, transform, **f2q_kwargs)
cisd_state = fermion_to_qubit_state(cisd_ground(mol), transform, **f2q_kwargs)

pham = fermion_to_qubit_operator(fham, transform, **f2q_kwargs)
p_const = pham.constant
# Make qubit hamiltonian traceless
pham = pham - p_const

cisd_energy = mol.cisd_energy - p_const - f_const

## 2. Pauli Symmetry

### 2-1. Find symmetry operators and objects

For a Hamiltonian
$$
    \hat{H} = \sum_j^{N_P} \alpha_j \hat{P}_j,
$$
find set of Pauli symmetry operators $\{\hat{S}_k\}$ such that
$$
    [\hat{S}_k, \hat{P}_j] = 0 \quad \forall k, j \quad \mathrm{thus, }\quad [\hat{H}, \hat{P}_j] = 0.
$$

Symmetry operators are selected with maximal mutual commutativity:
$$
        \mathcal{S}_M=arg\,max_{\mathcal{S}}|\mathcal{S}|\quad \mathrm{s.t. }\quad
        \mathcal{S}\subseteq \{\hat{S}_k: \forall k\}, \quad [\hat{S}_k,
        \hat{S}_l] = 0 \quad \forall \hat{S}_k, \hat{S}_l \in \mathcal{S}.
$$

Furthermore, the symmetry operators are diagonalized as X-type Pauli operators by applying the following Clifford transformation:
$$
    \hat{S}^{(X)}_k = \hat{C} \hat{S}_k \hat{C}^{\dagger} \quad \forall \hat{S}_k \in \mathcal{S}_M.
$$

Then, applying the Clifford transformation to the Hamiltonian allows the following commutation:
$$
    [\hat{S}^{(X)}_k, \hat{C}\hat{H}\hat{C}^{\dagger}] = 0.
$$

Consider the following unitary operator that maps each $\hat{S}_k^{(X)}$ to single qubit operator $\hat{Z}_{q(k)}$:
$$
    \hat{U}=\prod_k \frac{1}{2}(\hat{Z}_{q(k)}+\hat{S}_k^{(X)}),\quad \{\hat{Z}_{q(k)},\hat{S}^{(X)}_k\} = 0 \quad\mathrm{and}\quad [\hat{Z}_{q(k)}, \hat{S}_{l}] = 0 \quad \forall k\neq l.
$$

Then, the simulation Hamiltonian is block-diagonalized form as
$$
    \hat{H}_{sim} = \hat{U}\hat{C}\hat{H}\hat{C}^{\dagger}\hat{U}^{\dagger} = \sum_{p=0}^{2^{n_r}} \ket{p}\bra{p}\hat{H}_{\mathrm{red},p},
$$
where $n_r$ is the number of reduced qubits.

In [18]:
symm_operators, clifford_list, reduced_qubits, unitary = find_pauli_symmetry(pham, n_qubits)
print(f"symm_operators =\n\t"+"\n\t".join(tableau_to_pauli(symm_operators).__str__().split('\n')))
print(f"symm_clifford  = {clifford_list}")
print(f"reduced_qubits = {reduced_qubits}")
print(f"unitary =\n\t" + "\n\t".join(unitary.__str__().split('\n')))

symm_operators =
	[1.0 [X0], 1.0 [X1], 1.0 [X2]]
symm_clifford  = ['H_0', 'H_1', 'H_7', 'QSW_2_7', 'CZ_0_4', 'CZ_0_6', 'CZ_0_7', 'CZ_1_5', 'H_0', 'H_1', 'H_2', 'H_0', 'H_1', 'H_2']
reduced_qubits = [0, 1, 2]
unitary =
	0.35355339059327384 [X0 X1 X2] +
	0.35355339059327384 [X0 X1 Z2] +
	0.35355339059327384 [X0 Z1 X2] +
	0.35355339059327384 [X0 Z1 Z2] +
	0.35355339059327384 [Z0 X1 X2] +
	0.35355339059327384 [Z0 X1 Z2] +
	0.35355339059327384 [Z0 Z1 X2] +
	0.35355339059327384 [Z0 Z1 Z2]


In [19]:
qubit_reduced_hamiltonian = qubit_reduction_operator(pham, n_qubits, clifford_list, reduced_qubits, unitary)
print(f"{len(reduced_qubits)} qubits are reduced, thus the original hamiltonian is decomposed to {2 ** len(reduced_qubits)} {n_qubits - len(reduced_qubits)}-qubit Hamiltonians:")
for k, red_op in qubit_reduced_hamiltonian.items():
    print(f"qubit-parity pair: {k}")
    print(f"red_op =\n\t" + '\n\t'.join(red_op.__str__().split('\n')[:3])+"\n\t...")

3 qubits are reduced, thus the original hamiltonian is decomposed to 8 5-qubit Hamiltonians:
qubit-parity pair: ((0, -1), (1, -1), (2, -1))
red_op =
	-0.011706349170573073 [X0 X1 Y2 Y3 Z4] +
	0.026450672123986833 [X0 X1 Y2 Z3 Y4] +
	0.025863272971678477 [X0 X1 Z2 X3] +
	...
qubit-parity pair: ((0, -1), (1, -1), (2, 1))
red_op =
	0.011706349170573073 [X0 X1 Y2 Y3 Z4] +
	0.026450672123986833 [X0 X1 Y2 Z3 Y4] +
	0.025863272971678477 [X0 X1 Z2 X3] +
	...
qubit-parity pair: ((0, -1), (1, 1), (2, -1))
red_op =
	-0.011706349170573073 [X0 X1 Y2 Y3 Z4] +
	-0.026450672123986833 [X0 X1 Y2 Z3 Y4] +
	0.025863272971678477 [X0 X1 Z2 X3] +
	...
qubit-parity pair: ((0, -1), (1, 1), (2, 1))
red_op =
	0.011706349170573073 [X0 X1 Y2 Y3 Z4] +
	-0.026450672123986833 [X0 X1 Y2 Z3 Y4] +
	0.025863272971678477 [X0 X1 Z2 X3] +
	...
qubit-parity pair: ((0, 1), (1, -1), (2, -1))
red_op =
	0.011706349170573073 [X0 X1 Y2 Y3 Z4] +
	-0.026450672123986833 [X0 X1 Y2 Z3 Y4] +
	0.025863272971678477 [X0 X1 Z2 X3] +
	...
qu

Let us compare the eigenvalues and eigenvectors of $\hat{H}$ and $\hat{H}_{sim}$.

In [20]:
dim_ham = 2 ** n_qubits
pham_eval, pham_evec = diagonalization(pham, n_qubits, sparse_eig=False)

n_red_qubits = n_qubits - len(reduced_qubits)
dim_red_ham = 2 ** n_red_qubits

red_pham_eval, red_pham_evec = np.zeros(dim_ham), np.zeros((dim_red_ham, dim_ham), dtype=complex)
parity_list = list()
for idx, (parity, red_op) in enumerate(qubit_reduced_hamiltonian.items()):
    cur_red_pham_eval, cur_red_pham_evec = diagonalization(red_op, n_red_qubits, sparse_eig=False)
    red_pham_eval[idx * dim_red_ham:(idx + 1) * dim_red_ham] = cur_red_pham_eval
    red_pham_evec[:, idx * dim_red_ham:(idx + 1) * dim_red_ham] = cur_red_pham_evec
    parity_list += [parity] * dim_red_ham

a_sort = np.argsort(pham_eval)
pham_eval, pham_evec = pham_eval[a_sort], pham_evec[:, a_sort]

a_sort = np.argsort(red_pham_eval)
red_pham_eval, red_pham_evec = red_pham_eval[a_sort], red_pham_evec[:, a_sort]
parity_list = [parity_list[idx] for idx in a_sort]


In [21]:
assert np.allclose(pham_eval, red_pham_eval)

degenerated_indices = [[0]]
for idx in range(1, dim_ham):
    if np.isclose(pham_eval[idx-1], pham_eval[idx]):
        degenerated_indices[-1].append(idx)
    else:
        degenerated_indices.append([idx])

# Compare each eigenspace
for indices in degenerated_indices:
    degen_parity = list(set([parity_list[idx] for idx in indices]))
    red_pham_espace_1 = {parity: np.zeros((dim_red_ham, dim_red_ham), dtype=complex)
                       for parity in degen_parity}
    red_pham_espace_2 = {parity: np.zeros((dim_red_ham, dim_red_ham), dtype=complex)
                       for parity in degen_parity}

    n_degen = len(indices)
    for idx in indices:
        red_states, weights = qubit_reduction_state(pham_evec[:, idx], clifford_list, reduced_qubits, unitary)
        for parity in red_states.keys():
            red_pham_espace_1[parity] += (abs(weights[parity])**2 *
                                          np.outer(red_states[parity], red_states[parity].conjugate()))

        parity = parity_list[idx]
        red_pham_espace_2[parity] += np.outer(red_pham_evec[:, idx], red_pham_evec[:, idx].conjugate())

    assert sorted(red_pham_espace_1.keys()) == sorted(red_pham_espace_2.keys())
    for parity in red_pham_espace_1.keys():
        assert np.allclose(red_pham_espace_1[parity], red_pham_espace_2[parity])
